# UrbanRisk AI — M3: Chennai Traffic Congestion Prediction
**Milestone 3 of 9** — Near-term congestion-risk model, built around a data constraint that's actually worse than M2's.

### Read this before anything else in the notebook
The original spec asked for congestion predicted **+30 / +60 / +120 minutes ahead**. That resolution implies we
have minute-level historical traffic speed data to learn from. **We don't, and nobody does for free, for Chennai:**

| Option | Why it doesn't give us a free historical minute-level series |
|---|---|
| Google Roads/Directions API (traffic_model) | Requires a billed account — not a free tier, just free credits |
| TomTom / HERE / Mapbox traffic-flow APIs | Free tiers exist but are **live-only** — a real snapshot of *right now*, not a rewindable history |
| Uber Movement | Discontinued |
| OpenTraffic | Discontinued |
| Chennai Traffic Police / CMDA | Publish corridor studies and peak-hour reports, not an open live/historical feed |

Rather than fabricate 30/60/120-minute deltas we have no ground truth for, this notebook makes the same honesty
call M2 made for flooding: **define a transparent, documented hourly proxy, and keep every piece of real
evidence we do have as a separate, clearly-labeled feature — never blended into a label that pretends to be
measured congestion.**

Concretely, this notebook predicts `congestion_risk_next_1h` and `congestion_risk_next_2h` (not 30/120 min —
that's the deviation from the original spec, made explicitly and documented here) using:

| Source | What it is | Role |
|---|---|---|
| **OSM/Overpass road network** (same API M2 used for water bodies) | Real road class, lane count, junction density per zone | Static congestion-*vulnerability* feature |
| **Hand-compiled chronic-congestion corridor list** (Kathipara, OMR IT corridor, GST Road, Anna Salai, Koyambedu, Vadapalani — from published Chennai traffic studies) | Real, named, documented bottlenecks with real coordinates | Static "known hotspot" distance feature — same role GCC inundation points played in M2 |
| **Open-Meteo historical rainfall** (same source/method as M2) | Real hourly rainfall | Dynamic feature — Chennai rain → waterlogging → congestion is a documented causal chain, not a guess |
| **Calendar (hour, day-of-week, weekend)** | Deterministic, not fetched | The actual dominant, well-documented driver of urban congestion |

### The target label — same warning as M2
`congestion_risk_next_{1,2}h` is defined as: **1 if the hour falls in a published Chennai peak-traffic window
(morning 08:00–11:00, evening 17:30–21:00 on weekdays), boosted when rainfall in that window crosses the same
IMD heavy-rain threshold used in M2, 0 otherwise.** This is a **documented decision-support heuristic**, not a
measured congestion outcome. A Precision/Recall/F1 number here tells you "how well does the model recover a
known rush-hour + rain pattern," not "how well does it predict actual road speed." Say that sentence before
quoting any metric from this notebook.

If you later get a free TomTom/HERE API key, Section 8 shows where to splice in real live snapshots as a small
genuine validation set — the same way M2 kept GCC data as real, separate evidence.


## 1. Setup — same zone grid as M1/M2

In [ ]:
!pip install -q xgboost shap scikit-learn pandas numpy requests matplotlib

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from datetime import datetime, timedelta

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (12, 4)

END_DATE = (datetime.utcnow() - timedelta(days=5)).strftime("%Y-%m-%d")
START_DATE = (datetime.utcnow() - timedelta(days=5) - timedelta(days=730)).strftime("%Y-%m-%d")
print("Pulling weather data from", START_DATE, "to", END_DATE)

ZONES = [
    ("CHN_CENTRAL",   "Chennai Central",    13.0827,  80.2707),
    ("CHN_MANALI",    "Manali",             13.1631,  80.2586),
    ("CHN_KODUNGYR",  "Kodungaiyur",        13.1275,  80.2470),
    ("CHN_ROYAPRM",   "Royapuram",          13.1143,  80.2947),
    ("CHN_WSHRMNPT",  "Washermanpet",       13.1167,  80.2833),
    ("CHN_PERAMBUR",  "Perambur",           13.1102,  80.2323),
    ("CHN_KOYAMBDU",  "Koyambedu",          13.0694,  80.1948),
    ("CHN_ANNANGR",   "Anna Nagar",         13.0850,  80.2101),
    ("CHN_AMBATTUR",  "Ambattur",           13.1143,  80.1548),
    ("CHN_AVADI",     "Avadi",              13.1147,  80.0970),
    ("CHN_NUNGMBKM",  "Nungambakkam",       13.0569,  80.2425),
    ("CHN_TNAGAR",    "T Nagar",            13.0418,  80.2341),
    ("CHN_MYLAPORE",  "Mylapore",           13.0339,  80.2619),
    ("CHN_ADYAR",     "Adyar",              13.0067,  80.2570),
    ("CHN_GUINDY",    "Guindy",             13.0067,  80.2206),
    ("CHN_VELACHRY",  "Velachery",          12.9756,  80.2200),
    ("CHN_PERUNGDI",  "Perungudi",          12.9647,  80.2422),
    ("CHN_SHOLINGL",  "Sholinganallur/OMR", 12.9010,  80.2279),
    ("CHN_TAMBARAM",  "Tambaram",           12.9249,  80.1000),
    ("CHN_PALLAVRM",  "Pallavaram",         12.9675,  80.1491),
]
zones_df = pd.DataFrame(ZONES, columns=["zone_id", "zone_name", "latitude", "longitude"])
print(f"{len(zones_df)} zones (same grid as M1/M2)")


## 2. Real static evidence — road network structure (OSM / Overpass)
Same Overpass API M2 used for water bodies, with the same retry + mirror-fallback pattern, this time querying road segments and junctions within each zone's radius.

In [ ]:
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
]

ROAD_RADIUS_M = 1500  # matches the 1.5km radius M2 used for inundation-point vulnerability

def overpass_query(query, max_retries=3):
    last_err = None
    for mirror in OVERPASS_MIRRORS:
        for attempt in range(max_retries):
            try:
                r = requests.post(mirror, data={"data": query}, timeout=60)
                r.raise_for_status()
                return r.json()
            except Exception as e:
                last_err = e
                time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"All Overpass mirrors failed: {last_err}")

def fetch_road_features(lat, lon, radius_m=ROAD_RADIUS_M):
    q = (
        "[out:json][timeout:50];\n"
        "(\n"
        f'  way["highway"~"motorway|trunk|primary|secondary|tertiary"](around:{radius_m},{lat},{lon});\n'
        f'  node["highway"="traffic_signals"](around:{radius_m},{lat},{lon});\n'
        ");\n"
        "out tags;\n"
    )
    data = overpass_query(q)
    elements = data.get("elements", [])
    major_roads = sum(1 for e in elements if e.get("type") == "way")
    signals = sum(1 for e in elements if e.get("type") == "node")
    return major_roads, signals

road_records = []
for _, zone in zones_df.iterrows():
    try:
        major_roads, signals = fetch_road_features(zone["latitude"], zone["longitude"])
    except Exception as e:
        print(f"  Overpass failed for {zone['zone_name']}: {e} — recording as 0 (flagged, not silently dropped)")
        major_roads, signals = 0, 0
    road_records.append({
        "zone_id": zone["zone_id"],
        "major_road_segments": major_roads,
        "traffic_signal_count": signals,
    })
    time.sleep(1)  # be polite to the shared Overpass mirrors

road_features_df = pd.DataFrame(road_records)
road_features_df["road_density_score"] = (
    road_features_df["major_road_segments"] + road_features_df["traffic_signal_count"]
)
road_features_df.head(20)


## 3. Real static evidence — documented chronic-congestion corridors
Hand-compiled from published Chennai traffic studies and Chennai Traffic Police corridor reports (same honesty standard as M2's hand-verified water-body list: real, named, coordinate-checked locations, not an exhaustive live feed).

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# Real, named chronic-bottleneck locations, repeatedly documented in Chennai traffic studies/news coverage.
# Not exhaustive — major, well-established hotspots only.
CONGESTION_HOTSPOTS = [
    ("Kathipara Flyover/Junction", 13.0106, 80.2064),
    ("Koyambedu Junction",          13.0694, 80.1948),
    ("Vadapalani Junction",         13.0503, 80.2129),
    ("Anna Salai / Teynampet",      13.0428, 80.2469),
    ("T Nagar Pondy Bazaar",        13.0418, 80.2341),
    ("OMR IT Corridor (Sholinganallur)", 12.9010, 80.2279),
    ("GST Road (Pallavaram)",       12.9675, 80.1491),
    ("Perambur Barrier",            13.1102, 80.2323),
]
hotspots_df = pd.DataFrame(CONGESTION_HOTSPOTS, columns=["hotspot_name", "latitude", "longitude"])

hotspot_records = []
for _, zone in zones_df.iterrows():
    dists = haversine_km(zone["latitude"], zone["longitude"],
                          hotspots_df["latitude"].values, hotspots_df["longitude"].values)
    hotspot_records.append({
        "zone_id": zone["zone_id"],
        "distance_to_congestion_hotspot_km": float(dists.min()),
    })

hotspot_features_df = pd.DataFrame(hotspot_records)
hotspot_features_df.head(20)


## 4. Real dynamic data — hourly rainfall (same source/method as M2)

In [ ]:
def fetch_hourly_weather(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "precipitation,temperature_2m,relative_humidity_2m",
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()["hourly"]
    return pd.DataFrame({
        "timestamp": pd.to_datetime(j["time"]),
        "rainfall_mm": j["precipitation"],
        "temperature_c": j["temperature_2m"],
        "humidity_pct": j["relative_humidity_2m"],
    })

weather_frames = []
for _, zone in zones_df.iterrows():
    wdf = fetch_hourly_weather(zone["latitude"], zone["longitude"], START_DATE, END_DATE)
    wdf["zone_id"] = zone["zone_id"]
    wdf["zone_name"] = zone["zone_name"]
    weather_frames.append(wdf)
    time.sleep(0.3)

weather_df = pd.concat(weather_frames, ignore_index=True)
weather_df["rainfall_6h_sum"] = (
    weather_df.sort_values("timestamp")
    .groupby("zone_id")["rainfall_mm"]
    .transform(lambda s: s.rolling(6, min_periods=1).sum())
)
print(f"{len(weather_df):,} zone-hours of real Open-Meteo data")
weather_df.head()


## 5. The proxy target — read this before trusting any metric below
`congestion_risk_next_{h}h` = 1 if the target hour falls in a **documented Chennai weekday peak window**
(08:00–11:00 or 17:30–21:00) **or** rainfall in that hour crosses the IMD Heavy Rain threshold used in M2
(waterlogging-driven congestion is a well-documented, not invented, causal link), 0 otherwise.

This is deliberately simple and deliberately transparent — it is a decision-support heuristic built from two
published, citable facts (peak-hour windows, IMD rainfall thresholds), not a hidden or "learned" definition. The
model's job is to recover this pattern *using the static hotspot/road features as well as time and rain*, so
that Section 7's SHAP output can show whether the real corridor/road-network evidence adds anything over "it's
rush hour."


In [ ]:
IMD_HEAVY_RAIN_MM_24H = 64.5  # same published threshold used in M2

def is_weekday_peak(ts):
    if ts.weekday() >= 5:
        return False
    hour_min = ts.hour + ts.minute / 60
    return (8.0 <= hour_min <= 11.0) or (17.5 <= hour_min <= 21.0)

def build_congestion_target(df, horizon_hours):
    df = df.sort_values(["zone_id", "timestamp"]).copy()
    shifted_ts = df["timestamp"] + pd.to_timedelta(horizon_hours, unit="h")
    peak_flag = shifted_ts.apply(is_weekday_peak)

    # rainfall in the target hour, scaled to the same proportional logic M2 used for its flood threshold
    rain_threshold_scaled = IMD_HEAVY_RAIN_MM_24H * (horizon_hours / 24.0)
    future_rain = (
        df.groupby("zone_id")["rainfall_mm"]
        .shift(-horizon_hours)
        .fillna(0)
    )
    rain_flag = future_rain >= rain_threshold_scaled

    return (peak_flag.values | rain_flag.values).astype(int)

weather_df = weather_df.sort_values(["zone_id", "timestamp"]).reset_index(drop=True)
weather_df["congestion_risk_next_1h"] = build_congestion_target(weather_df, 1)
weather_df["congestion_risk_next_2h"] = build_congestion_target(weather_df, 2)

for h in [1, 2]:
    rate = weather_df[f"congestion_risk_next_{h}h"].mean()
    print(f"Positive rate, next_{h}h: {rate:.1%}")


## 6. Assemble the modeling dataset — real static features + real rainfall + calendar

In [ ]:
master_df = weather_df.merge(road_features_df, on="zone_id", how="left")
master_df = master_df.merge(hotspot_features_df, on="zone_id", how="left")

master_df["hour"] = master_df["timestamp"].dt.hour
master_df["day_of_week"] = master_df["timestamp"].dt.dayofweek
master_df["is_weekend"] = (master_df["day_of_week"] >= 5).astype(int)
master_df["zone_id"] = master_df["zone_id"].astype("category")

feature_cols = [
    "zone_id", "hour", "day_of_week", "is_weekend",
    "rainfall_mm", "rainfall_6h_sum", "temperature_c", "humidity_pct",
    "major_road_segments", "traffic_signal_count", "road_density_score",
    "distance_to_congestion_hotspot_km",
]
target_cols = ["congestion_risk_next_1h", "congestion_risk_next_2h"]

master_df = master_df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)

# chronological split — no shuffling, same discipline as M2
master_df = master_df.sort_values("timestamp").reset_index(drop=True)
n = len(master_df)
train_df = master_df.iloc[: int(n * 0.7)]
val_df   = master_df.iloc[int(n * 0.7): int(n * 0.85)]
test_df  = master_df.iloc[int(n * 0.85):]

print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
X_train, y_train = train_df[feature_cols], train_df[target_cols]
X_val,   y_val   = val_df[feature_cols],   val_df[target_cols]
X_test,  y_test  = test_df[feature_cols],  test_df[target_cols]


## 7. Models — LogReg / Random Forest / XGBoost, per horizon, with class-imbalance handling

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from xgboost import XGBClassifier

lr_feature_cols = [c for c in feature_cols if c != "zone_id"]

def evaluate_clf(y_true, y_prob, threshold, label):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if y_true.nunique() > 1 else float("nan"),
        "pr_auc": average_precision_score(y_true, y_prob) if y_true.nunique() > 1 else float("nan"),
    }

clf_results = {}
trained_models = {}

for h in [1, 2]:
    target_col = f"congestion_risk_next_{h}h"
    y_train_h, y_val_h, y_test_h = y_train[target_col], y_val[target_col], y_test[target_col]

    scaler = StandardScaler()
    X_train_lr = scaler.fit_transform(X_train[lr_feature_cols])
    X_test_lr = scaler.transform(X_test[lr_feature_cols])
    lr = LogisticRegression(max_iter=1000, class_weight="balanced")
    lr.fit(X_train_lr, y_train_h)
    y_prob_lr = lr.predict_proba(X_test_lr)[:, 1]
    clf_results[f"logreg_{h}h"] = evaluate_clf(y_test_h, y_prob_lr, 0.5, f"LogReg +{h}h")

    X_train_rf = pd.get_dummies(X_train, columns=["zone_id"])
    X_test_rf = pd.get_dummies(X_test, columns=["zone_id"]).reindex(columns=X_train_rf.columns, fill_value=0)
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_train_rf, y_train_h)
    y_prob_rf = rf.predict_proba(X_test_rf)[:, 1]
    clf_results[f"rf_{h}h"] = evaluate_clf(y_test_h, y_prob_rf, 0.5, f"RandomForest +{h}h")

    scale_pos_weight = (y_train_h == 0).sum() / max((y_train_h == 1).sum(), 1)
    X_train_xgb = X_train.copy(); X_train_xgb["zone_id"] = X_train_xgb["zone_id"].cat.codes
    X_val_xgb = X_val.copy(); X_val_xgb["zone_id"] = X_val_xgb["zone_id"].cat.codes
    X_test_xgb = X_test.copy(); X_test_xgb["zone_id"] = X_test_xgb["zone_id"].cat.codes
    xgb = XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric="aucpr", scale_pos_weight=scale_pos_weight, early_stopping_rounds=30,
    )
    xgb.fit(X_train_xgb, y_train_h, eval_set=[(X_val_xgb, y_val[target_col])], verbose=False)
    y_prob_xgb = xgb.predict_proba(X_test_xgb)[:, 1]
    clf_results[f"xgboost_{h}h"] = evaluate_clf(y_test_h, y_prob_xgb, 0.5, f"XGBoost +{h}h")

    trained_models[h] = {"logreg": lr, "rf": rf, "xgboost": xgb, "scaler": scaler,
                          "train_rf_cols": X_train_rf.columns}

pd.DataFrame(clf_results).T


## 8. Precision-Recall curves
(More informative than ROC here — positives are rush-hour/rain windows, a minority of all hours.)

**Optional real-data upgrade:** if you get a free TomTom or HERE traffic-flow API key, poll it for these same 20 zone centers a few times a day for a week or two. That gives you a small set of *genuinely measured* speed-vs-freeflow ratios — plug them in here as a held-out sanity check against the proxy (the same role the GCC inundation points played as real evidence in M2), rather than as the training label itself, since a few weeks of live polling still isn't a historical series.

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, h in zip(axes, [1, 2]):
    target_col = f"congestion_risk_next_{h}h"
    y_true = y_test[target_col]
    for name, key in [("LogReg", "logreg"), ("RandomForest", "rf"), ("XGBoost", "xgboost")]:
        model = trained_models[h][key]
        if key == "logreg":
            y_prob = model.predict_proba(trained_models[h]["scaler"].transform(X_test[lr_feature_cols]))[:, 1]
        elif key == "rf":
            X_test_rf = pd.get_dummies(X_test, columns=["zone_id"]).reindex(
                columns=trained_models[h]["train_rf_cols"], fill_value=0)
            y_prob = model.predict_proba(X_test_rf)[:, 1]
        else:
            X_test_xgb = X_test.copy(); X_test_xgb["zone_id"] = X_test_xgb["zone_id"].cat.codes
            y_prob = model.predict_proba(X_test_xgb)[:, 1]
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ax.plot(recall, precision, label=name)
    ax.set_title(f"Precision-Recall — congestion_risk_next_{h}h")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend()
plt.tight_layout()
plt.show()


## 9. SHAP explanation — XGBoost, +1h horizon
Check where `distance_to_congestion_hotspot_km`, `road_density_score` and `traffic_signal_count` (the real static evidence) rank against `hour`/`day_of_week` (which mechanically define most of the label). If they rank low, the model is mostly just relearning the rush-hour clock — worth reporting honestly, not burying.

In [ ]:
import shap

h = 1
X_test_shap = X_test.copy()
X_test_shap["zone_id"] = X_test_shap["zone_id"].cat.codes

explainer = shap.TreeExplainer(trained_models[h]["xgboost"])
shap_values = explainer(X_test_shap)

shap.summary_plot(shap_values, X_test_shap, max_display=15, show=True)


## 10. Per-zone risk snapshot

In [ ]:
h = 1
snapshot = test_df.copy()
X_test_xgb = X_test.copy(); X_test_xgb["zone_id"] = X_test_xgb["zone_id"].cat.codes
snapshot["predicted_congestion_prob_1h"] = trained_models[h]["xgboost"].predict_proba(X_test_xgb)[:, 1]

zone_summary = snapshot.groupby("zone_name").agg(
    mean_predicted_prob=("predicted_congestion_prob_1h", "mean"),
    distance_to_congestion_hotspot_km=("distance_to_congestion_hotspot_km", "first"),
    road_density_score=("road_density_score", "first"),
).sort_values("mean_predicted_prob", ascending=False)

zone_summary


## 11. Save models & metadata

In [ ]:
import joblib
import json
import os

os.makedirs("models", exist_ok=True)

for h in [1, 2]:
    joblib.dump(trained_models[h]["xgboost"], f"models/congestion_xgboost_{h}h.joblib")
    joblib.dump(trained_models[h]["rf"], f"models/congestion_rf_{h}h.joblib")
    joblib.dump(trained_models[h]["logreg"], f"models/congestion_logreg_{h}h.joblib")

road_features_df.to_csv("models/traffic_zones_road_features.csv", index=False)
hotspot_features_df.to_csv("models/traffic_zones_hotspot_distance.csv", index=False)

metadata = {
    "deviation_from_original_spec": (
        "Original spec asked for +30/+60/+120 minute horizons. No free historical minute-level traffic "
        "dataset exists for Chennai (or generally), so this milestone predicts +1h/+2h instead, using an "
        "hourly-resolution proxy target. This is documented here rather than silently substituted."
    ),
    "target_definition": (
        "PROXY LABEL, not measured congestion. congestion_risk_next_{h}h = 1 if the target hour falls in a "
        "documented Chennai weekday peak window (08:00-11:00 or 17:30-21:00) OR rainfall in that hour meets "
        "the same IMD Heavy Rain threshold (scaled) used in M2. Sources: published Chennai peak-hour patterns "
        "+ India Meteorological Department rainfall classification."
    ),
    "static_feature_sources": {
        "road_network": "OpenStreetMap via Overpass API — major road segment count + traffic signal count within 1.5km of zone center",
        "congestion_hotspots": "Hand-compiled list of documented chronic-congestion corridors from published Chennai traffic studies (Kathipara, Koyambedu, Vadapalani, Anna Salai, T Nagar, OMR IT corridor, GST Road, Perambur) — real coordinates, not exhaustive",
    },
    "rainfall_source": "Open-Meteo Historical Weather Archive (same as M2)",
    "feature_columns": feature_cols,
    "target_columns": target_cols,
    "test_metrics": clf_results,
    "train_range": [str(train_df["timestamp"].min()), str(train_df["timestamp"].max())],
    "test_range": [str(test_df["timestamp"].min()), str(test_df["timestamp"].max())],
    "optional_upgrade_path": (
        "If a free TomTom/HERE API key becomes available, poll live traffic-flow for these 20 zone centers "
        "over 1-2 weeks and use the measured speed-vs-freeflow ratio as a held-out sanity check against this "
        "proxy's predictions — not as a training label, since a couple weeks of live polling is still not a "
        "historical series."
    ),
}

with open("models/traffic_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved M3 traffic congestion models + metadata to ./models/")
print(json.dumps(metadata["target_definition"], indent=2))
print(json.dumps(metadata["deviation_from_original_spec"], indent=2))
